# Validate Organism Mapping



Validate reviewed organism terminology mappings

Source organism labels remain unchanged in the upstream
clinical facts. Reviewed mappings provide standardized
identities for explicitly selected analytical organisms.

These checks measure mapping coverage and identify cases
where multiple source labels within one culture map to the
same standardized organism.

1. Mapping coverage by cohort.

In [1]:
%sql
select
    cohort_id,

    count(*) as culture_organism_rows,

    sum(
        case
            when mapping_status = 'reviewed' then 1
            else 0
        end
    ) as reviewed_mapping_rows,

    round(
        100.0
        * sum(
            case
                when mapping_status = 'reviewed' then 1
                else 0
            end
        )
        / count(*),
        1
    ) as reviewed_mapping_pct,

    count(distinct organism_source) as source_organism_labels,

    count(
        distinct case
            when mapping_status = 'reviewed'
                then organism_source
        end
    ) as reviewed_source_labels,

    count(
        distinct case
            when include_primary_summary = 1
                then organism_standardized
        end
    ) as primary_standardized_organisms

from workspace.micro_dev.int_mapped_cohort_culture_organism

group by cohort_id

order by cohort_id;

,cohort_id,culture_organism_rows,reviewed_mapping_rows,reviewed_mapping_pct,source_organism_labels,reviewed_source_labels,primary_standardized_organisms
0,blood,13651,9033,66.2,238,22,12
1,respiratory,22411,17491,78.0,167,20,12
2,urine,97184,81982,84.4,142,20,12


2. Most frequent source labels that remain unmapped.

This helps prioritize future terminology review without
automatically assigning biological identities.

In [2]:
%sql
select
    cohort_id,
    organism_source,
    count(*) as culture_organism_rows,
    count(distinct patient_key) as patients

from workspace.micro_dev.int_mapped_cohort_culture_organism

where mapping_status = 'unmapped'

group by
    cohort_id,
    organism_source

order by
    cohort_id,
    culture_organism_rows desc;

,cohort_id,organism_source,culture_organism_rows,patients
0,blood,COAG NEGATIVE STAPHYLOCOCCUS,1913,1247
1,blood,STREPTOCOCCUS MITIS GROUP,262,251
2,blood,STREPTOCOCCUS ANGINOSUS GROUP,132,129
3,blood,STREPTOCOCCUS PYOGENES (GROUP A),122,120
4,blood,BACTEROIDES FRAGILIS,96,96
5,blood,VIRIDANS GROUP STREPTOCOCCI,86,83
6,blood,STREPTOCOCCUS DYSGALACTIAE,84,81
7,blood,ENTEROBACTER CLOACAE,63,62
8,blood,CITROBACTER FREUNDII COMPLEX,59,56
9,blood,STREPTOCOCCUS MITIS/STREPTOCOCCUS ORALIS,59,57


3. Detect mapping collisions within a culture.

A collision occurs when more than one distinct source
organism label from the same culture maps to the same
standardized organism.

These must be reviewed before collapsing to a standardized
culture-organism grain.

In [3]:
%sql
select
    cohort_id,
    culture_key,
    organism_standardized,

    count(distinct organism_source) as source_label_count,

    array_sort(
        collect_set(organism_source)
    ) as source_labels

from workspace.micro_dev.int_mapped_cohort_culture_organism

where include_primary_summary = 1

group by
    cohort_id,
    culture_key,
    organism_standardized

having count(distinct organism_source) > 1

order by
    source_label_count desc,
    cohort_id,
    organism_standardized;

,cohort_id,culture_key,organism_standardized,source_label_count,source_labels
0,respiratory,1fb458768ff7a678418489c64ad5b67b,Staphylococcus aureus,3,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
1,blood,d92862448e0fec67728ae7f30a0bc7af,Enterococcus faecium,2,"[ENTEROCOCCUS FAECIUM, ENTEROCOCCUS FAECIUM - VANCO RESISTANT]"
2,blood,c671c06a5e71de36b8393b7282dd96d1,Escherichia coli,2,"[ESCHERICHIA COLI, ESCHERICHIA COLI (CARBAPENEM RESISTANT)]"
3,blood,7e1c3012df06f03eccee35e98f086cb2,Staphylococcus aureus,2,"[STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
4,blood,a49634d47eae39e92da83daef8d5d1c4,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
5,blood,a22fd81033245e47e47b0aac5a5125cc,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
6,blood,6811eac316f0b8e528a80c41a56e8e15,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)]"
7,blood,c4fa8eb94355c019f9079f6804ac2263,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPHYLOCOCCUS AUREUS]"
8,blood,979280069e73610941ba700d3acc4c6a,Staphylococcus aureus,2,"[STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE), STAPHYLOCOCCUS AUREUS]"
9,blood,6021cc9118c63001a2b31a1325995dcf,Staphylococcus aureus,2,"[STAPH AUREUS {MRSA}, STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)]"


4. Summarize mapping collisions by cohort and standardized
organism.

In [4]:
%sql
with collisions as (

    select
        cohort_id,
        culture_key,
        organism_standardized

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where include_primary_summary = 1

    group by
        cohort_id,
        culture_key,
        organism_standardized

    having count(distinct organism_source) > 1

)

select
    cohort_id,
    organism_standardized,
    count(*) as culture_orders_with_multiple_source_labels

from collisions

group by
    cohort_id,
    organism_standardized

order by
    culture_orders_with_multiple_source_labels desc,
    cohort_id,
    organism_standardized;

,cohort_id,organism_standardized,culture_orders_with_multiple_source_labels
0,respiratory,Pseudomonas aeruginosa,1520
1,respiratory,Staphylococcus aureus,200
2,urine,Pseudomonas aeruginosa,52
3,blood,Staphylococcus aureus,9
4,urine,Enterobacter cloacae complex,3
5,urine,Klebsiella pneumoniae,2
6,blood,Enterococcus faecium,1
7,blood,Escherichia coli,1
8,respiratory,Enterobacter cloacae complex,1
9,urine,Escherichia coli,1


5. Verify that every organism selected for the primary
summary has a standardized identity.

Expected result: zero rows.

In [5]:
%sql
select
    organism_source,
    organism_standardized,
    include_primary_summary,
    mapping_status

from workspace.micro_dev.int_mapped_cohort_culture_organism

where include_primary_summary = 1
  and organism_standardized is null;

,organism_source,organism_standardized,include_primary_summary,mapping_status
